In [1]:
import json

knowledge_base = {
    "bearing_outer_race": {
        "signature": {"rms_threshold": 0.25, "kurtosis_threshold": 4.0},
        "hypothesis": "Outer-race bearing degradation (spalling or wear)",
        "severity": "HIGH",
        "actions": [
            "Immediate: Schedule vibration inspection within 24 hours.",
            "Check lubrication levels and contamination.",
            "Inspect outer race for spalling or pitting."
        ],
        "parts": ["Bearing SKF-6205-2RS", "High-temp grease"],
        "downtime_hours": 2.0,
        "manual_ref": "Section 4.2 - Bearing Inspection"
    },
    "general_vibration": {
        "signature": {"rms_threshold": 0.15, "kurtosis_threshold": 3.5},
        "hypothesis": "General vibration anomaly (possible misalignment or imbalance)",
        "severity": "MEDIUM",
        "actions": [
            "Monitor trend for 48 hours.",
            "Check mounting bolts for looseness."
        ],
        "parts": [],
        "downtime_hours": 0.5,
        "manual_ref": "Section 3.1 - Vibration Diagnostics"
    }
}

with open("knowledge_base.json", "w") as f:
    json.dump(knowledge_base, f, indent=4)

print("✅ knowledge_base.json created.")

In [2]:
%%writefile inference_engine.py
import paho.mqtt.client as mqtt
import json
import joblib
import numpy as np
from datetime import datetime

# Load model and knowledge base
model = joblib.load("isolation_forest_v1.joblib")
with open("knowledge_base.json", "r") as f:
    KB = json.load(f)

anomaly_streak = 0
persistence_threshold = 3
current_state = "NORMAL"

def diagnose_fault(rms, kurtosis):
    """Deterministic rule engine to map features to a maintenance action."""
    # Priority 1: Check for severe bearing fault
    if rms > KB["bearing_outer_race"]["signature"]["rms_threshold"] and \
       kurtosis > KB["bearing_outer_race"]["signature"]["kurtosis_threshold"]:
        return KB["bearing_outer_race"]
    
    # Priority 2: Check for general vibration
    elif rms > KB["general_vibration"]["signature"]["rms_threshold"]:
        return KB["general_vibration"]
    
    return None

def on_connect(client, userdata, flags, reason_code, properties):
    print("🧠 Inference Engine Connected. Listening...")
    client.subscribe("factory/motor_01/vibration")

def on_message(client, userdata, msg):
    global anomaly_streak, current_state
    payload = json.loads(msg.payload.decode())
    
    # Extract features (Rms, Kurtosis, Crest, Skew)
    features = np.array([payload["features"]])
    rms = features[0][0]
    kurtosis = features[0][1]
    
    prediction = model.predict(features)[0]
    
    if prediction == -1:
        anomaly_streak += 1
    else:
        anomaly_streak = 0
        
    if anomaly_streak >= persistence_threshold:
        if current_state != "CRITICAL":
            # Trigger the diagnostic engine
            diagnosis = diagnose_fault(rms, kurtosis)
            
            if diagnosis:
                alert_payload = {
                    "timestamp_utc": datetime.utcnow().isoformat() + "Z",
                    "asset_id": payload["sensor_id"],
                    "fault_hypothesis": diagnosis["hypothesis"],
                    "severity": diagnosis["severity"],
                    "evidence": {
                        "rms_current": round(float(rms), 3),
                        "kurtosis_current": round(float(kurtosis), 3)
                    },
                    "recommended_actions": diagnosis["actions"],
                    "required_parts": diagnosis["parts"],
                    "estimated_downtime_hours": diagnosis["downtime_hours"],
                    "manual_reference": diagnosis["manual_ref"]
                }
                
                print("\n🚨 CRITICAL ALERT & DIAGNOSIS 🚨")
                print(json.dumps(alert_payload, indent=4))
                print("-" * 50)
            else:
                print(f"🚨 CRITICAL ALERT! Unclassified anomaly on {payload['sensor_id']}.")
                
            current_state = "CRITICAL"
            
    elif anomaly_streak == 0 and current_state == "CRITICAL":
        print("✅ RECOVERY: Motor has returned to normal operation.")
        current_state = "NORMAL"

client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
client.on_connect = on_connect
client.on_message = on_message
client.connect("localhost", 1883, 60)
client.loop_forever()

In [3]:
import time

# 1. CLEAN SLATE
print("🧹 Cleaning up old processes...")
!pkill -f inference_engine.py
!pkill -f sensor_publisher.py
!pkill mosquitto
time.sleep(1)

# 2. START BROKER
print("🚀 Starting MQTT Broker...")
!apt-get install mosquitto mosquitto-clients -y > /dev/null
!mosquitto -d
time.sleep(1)

# 3. RUN PIPELINE
print("🧠 Starting Inference Engine in the background...")
!rm -f inference_output.log
!python -u inference_engine.py > inference_output.log 2>&1 &
time.sleep(2) # Wait for engine to connect

print("📡 Starting Sensor Publisher...")
!python sensor_publisher.py

print("\nWaiting for final messages to process...")
time.sleep(3)

print("\n--- 🚨 COPILOT DIAGNOSIS LOGS ---")
!cat inference_output.log

In [4]:
!pip install paho-mqtt

In [5]:
%%writefile sensor_publisher.py
import paho.mqtt.client as mqtt
import time
import json
import numpy as np
import scipy.io as sio
from scipy.stats import kurtosis, skew

client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
client.connect("localhost", 1883, 60)
print("📡 Robust Sensor Publisher Started. Streaming REAL data...")

normal_data = sio.loadmat('97.mat')['X097_DE_time'].flatten()
faulty_data = sio.loadmat('105.mat')['X105_DE_time'].flatten()

normal_segment = normal_data[:50000]
faulty_segment = faulty_data[:50000]
transition_length = 10000
transition = faulty_segment[:transition_length] * np.linspace(0, 1, transition_length)
r2f_timeline = np.concatenate([normal_segment, transition, faulty_segment[transition_length:]])

noise_level = 0.1 * np.std(normal_segment)
noisy_timeline = r2f_timeline + np.random.normal(0, noise_level, len(r2f_timeline))

window_size = 1024
step_size = 512

for i in range(0, len(noisy_timeline) - window_size, step_size):
    window = noisy_timeline[i : i + window_size]
    rms = float(np.sqrt(np.mean(window**2)))
    kurt = float(kurtosis(window))
    crest = float(np.max(np.abs(window)) / rms if rms > 0 else 0)
    skew_val = float(skew(window))
    
    payload = {
        "sensor_id": "MOTOR-01",
        "timestamp": time.time(),
        "features": [rms, kurt, crest, skew_val]
    }
    
    client.publish("factory/motor_01/vibration", json.dumps(payload))
    if i % (step_size * 20) == 0:
        print(f"Streaming... Sample {i}/{len(noisy_timeline)} | RMS: {rms:.3f}")
    time.sleep(0.05)

print("✅ Publisher finished streaming real degradation data.")

In [6]:
%%writefile sensor_publisher.py
import paho.mqtt.client as mqtt
import time
import json
import numpy as np
import scipy.io as sio
from scipy.stats import kurtosis, skew

client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
client.connect("localhost", 1883, 60)
print("📡 Robust Sensor Publisher Started. Streaming REAL data...")

normal_data = sio.loadmat('97.mat')['X097_DE_time'].flatten()
faulty_data = sio.loadmat('105.mat')['X105_DE_time'].flatten()

normal_segment = normal_data[:50000]
faulty_segment = faulty_data[:50000]
transition_length = 10000
transition = faulty_segment[:transition_length] * np.linspace(0, 1, transition_length)
r2f_timeline = np.concatenate([normal_segment, transition, faulty_segment[transition_length:]])

noise_level = 0.1 * np.std(normal_segment)
noisy_timeline = r2f_timeline + np.random.normal(0, noise_level, len(r2f_timeline))

window_size = 1024
step_size = 512

for i in range(0, len(noisy_timeline) - window_size, step_size):
    window = noisy_timeline[i : i + window_size]
    rms = float(np.sqrt(np.mean(window**2)))
    kurt = float(kurtosis(window))
    crest = float(np.max(np.abs(window)) / rms if rms > 0 else 0)
    skew_val = float(skew(window))
    
    payload = {
        "sensor_id": "MOTOR-01",
        "timestamp": time.time(),
        "features": [rms, kurt, crest, skew_val]
    }
    
    client.publish("factory/motor_01/vibration", json.dumps(payload))
    if i % (step_size * 20) == 0:
        print(f"Streaming... Sample {i}/{len(noisy_timeline)} | RMS: {rms:.3f}")
    time.sleep(0.05)

print("✅ Publisher finished streaming real degradation data.")

In [7]:
import os
import json
import joblib
import scipy.io as sio
import numpy as np
from scipy.stats import kurtosis, skew
from sklearn.ensemble import IsolationForest

# --- 1. Restore Knowledge Base ---
knowledge_base = {
    "bearing_outer_race": {
        "signature": {"rms_threshold": 0.25, "kurtosis_threshold": 4.0},
        "hypothesis": "Outer-race bearing degradation (spalling or wear)",
        "severity": "HIGH",
        "actions": [
            "Immediate: Schedule vibration inspection within 24 hours.",
            "Check lubrication levels and contamination.",
            "Inspect outer race for spalling or pitting."
        ],
        "parts": ["Bearing SKF-6205-2RS", "High-temp grease"],
        "downtime_hours": 2.0,
        "manual_ref": "Section 4.2 - Bearing Inspection"
    },
    "general_vibration": {
        "signature": {"rms_threshold": 0.15, "kurtosis_threshold": 3.5},
        "hypothesis": "General vibration anomaly (possible misalignment or imbalance)",
        "severity": "MEDIUM",
        "actions": ["Monitor trend for 48 hours.", "Check mounting bolts for looseness."],
        "parts": [],
        "downtime_hours": 0.5,
        "manual_ref": "Section 3.1 - Vibration Diagnostics"
    }
}
with open("knowledge_base.json", "w") as f:
    json.dump(knowledge_base, f, indent=4)

# --- 2. Restore Data & Model ---
if not os.path.exists('97.mat'):
    import urllib.request
    urllib.request.urlretrieve('https://engineering.case.edu/sites/default/files/97.mat', '97.mat')
    urllib.request.urlretrieve('https://engineering.case.edu/sites/default/files/105.mat', '105.mat')

normal_data = sio.loadmat('97.mat')['X097_DE_time'].flatten()

def extract_features(window):
    rms = np.sqrt(np.mean(window**2))
    return [rms, kurtosis(window), np.max(np.abs(window)) / rms if rms > 0 else 0, skew(window)]

X_train = []
window_size = 1024
step_size = 512
for i in range(0, 50000 - window_size, step_size):
    X_train.append(extract_features(normal_data[i : i + window_size]))

# Train and save
final_model = IsolationForest(contamination=0.01, random_state=42)
final_model.fit(np.array(X_train))
joblib.dump(final_model, "isolation_forest_v1.joblib")
print("✅ Model and Knowledge Base successfully restored!")

In [8]:
%%writefile inference_engine.py
import paho.mqtt.client as mqtt
import json
import joblib
import numpy as np
from datetime import datetime

# Load model and knowledge base
try:
    model = joblib.load("isolation_forest_v1.joblib")
    with open("knowledge_base.json", "r") as f:
        KB = json.load(f)
except Exception as e:
    print(f"❌ FATAL: Missing artifacts. Did you run Step 3.1? Error: {e}")
    exit(1)

anomaly_streak = 0
persistence_threshold = 3
current_state = "NORMAL"

def diagnose_fault(rms, kurtosis):
    """Deterministic rule engine to map features to a maintenance action."""
    if rms > KB["bearing_outer_race"]["signature"]["rms_threshold"] and \
       kurtosis > KB["bearing_outer_race"]["signature"]["kurtosis_threshold"]:
        return KB["bearing_outer_race"]
    
    elif rms > KB["general_vibration"]["signature"]["rms_threshold"]:
        return KB["general_vibration"]
    
    return None

def on_connect(client, userdata, flags, reason_code, properties):
    print("🧠 Inference Engine Connected. Listening...")
    client.subscribe("factory/motor_01/vibration")

def on_message(client, userdata, msg):
    global anomaly_streak, current_state
    payload = json.loads(msg.payload.decode())
    
    features = np.array([payload["features"]])
    rms = features[0][0]
    kurtosis = features[0][1]
    
    prediction = model.predict(features)[0]
    
    if prediction == -1:
        anomaly_streak += 1
    else:
        anomaly_streak = 0
        
    if anomaly_streak >= persistence_threshold:
        if current_state != "CRITICAL":
            diagnosis = diagnose_fault(rms, kurtosis)
            
            if diagnosis:
                alert_payload = {
                    "timestamp_utc": datetime.utcnow().isoformat() + "Z",
                    "asset_id": payload["sensor_id"],
                    "fault_hypothesis": diagnosis["hypothesis"],
                    "severity": diagnosis["severity"],
                    "evidence": {
                        "rms_current": round(float(rms), 3),
                        "kurtosis_current": round(float(kurtosis), 3)
                    },
                    "recommended_actions": diagnosis["actions"],
                    "required_parts": diagnosis["parts"],
                    "estimated_downtime_hours": diagnosis["downtime_hours"],
                    "manual_reference": diagnosis["manual_ref"]
                }
                
                print("\n🚨 CRITICAL ALERT & DIAGNOSIS 🚨")
                print(json.dumps(alert_payload, indent=4))
                print("-" * 50)
            else:
                print(f"🚨 CRITICAL ALERT! Unclassified anomaly on {payload['sensor_id']}.")
                
            current_state = "CRITICAL"
            
    elif anomaly_streak == 0 and current_state == "CRITICAL":
        print("✅ RECOVERY: Motor has returned to normal operation.")
        current_state = "NORMAL"

client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
client.on_connect = on_connect
client.on_message = on_message
client.connect("localhost", 1883, 60)
client.loop_forever()

In [9]:
import time

# 1. CLEAN SLATE (Kill any zombie processes)
print("🧹 Cleaning up old processes...")
!pkill -f inference_engine.py || true
!pkill -f sensor_publisher.py || true
!pkill mosquitto || true
time.sleep(1)

# 2. START BROKER
print("🚀 Starting MQTT Broker...")
!apt-get install mosquitto mosquitto-clients -y > /dev/null
!mosquitto -d
time.sleep(1)

# 3. RUN PIPELINE
print("🧠 Starting Inference Engine in the background...")
!rm -f inference_output.log
!python -u inference_engine.py > inference_output.log 2>&1 &
time.sleep(2) # Wait for engine to connect to broker

print("📡 Starting Sensor Publisher...")
!python sensor_publisher.py

print("\nWaiting for final messages to process...")
time.sleep(3)

print("\n--- 🚨 COPILOT DIAGNOSIS LOGS ---")
!cat inference_output.log

In [10]:
manual_content = """
BEARING MAINTENANCE MANUAL - SECTION 4.2
Fault: Outer-Race Bearing Degradation
Symptoms: High vibration RMS (>0.25g), elevated kurtosis (>4.0), spectral energy at BPFO frequency.
Immediate Action: Schedule vibration inspection within 24 hours.
Inspection Procedure: Remove bearing housing. Inspect outer race for spalling, pitting, or discoloration. Check lubrication for contamination (metal particles, water).
Corrective Action: Replace bearing with SKF-6205-2RS. Clean housing. Apply high-temp grease (Lithium complex, NLGI 2).
Estimated Downtime: 2.0 hours.
Required Parts: 1x Bearing SKF-6205-2RS, 50g High-temp grease.

SECTION 3.1
Fault: General Vibration Anomaly
Symptoms: Moderate RMS elevation (0.15g - 0.25g), normal kurtosis.
Immediate Action: Monitor trend for 48 hours.
Inspection Procedure: Check mounting bolts for looseness. Verify alignment with laser tool.
Estimated Downtime: 0.5 hours.
Required Parts: None.
"""
with open("maintenance_manual.txt", "w") as f:
    f.write(manual_content)
print("✅ Manual created.")

In [11]:
!pip install faiss-cpu sentence-transformers

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load a lightweight embedding model (runs on CPU)
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Chunk the manual (simple paragraph split)
with open("maintenance_manual.txt", "r") as f:
    text = f.read()
chunks = [chunk.strip() for chunk in text.split("\n\n") if chunk.strip()]

# Create embeddings and index
embeddings = embedder.encode(chunks)
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))

def retrieve_context(query: str, top_k: int = 1) -> str:
    query_vec = embedder.encode([query]).astype('float32')
    distances, indices = index.search(query_vec, top_k)
    return "\n---\n".join([chunks[i] for i in indices[0]])

print("✅ Retriever ready.")

In [12]:
from transformers import pipeline

# Load lightweight SLM on CPU
generator = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-135M-Instruct", device=-1)

def generate_recommendation(evidence: dict, context: str) -> str:
    prompt = f"""You are a maintenance copilot. Use ONLY the provided manual context to answer.
    
    Manual Context: {context}
    
    Sensor Evidence: {evidence}
    
    Based on the manual, provide:
    1. Fault Hypothesis
    2. Severity (HIGH/MEDIUM/LOW)
    3. Recommended Actions (bullet points)
    4. Required Parts
    5. Estimated Downtime
    
    Recommendation:"""
    
    output = generator(prompt, max_new_tokens=150, do_sample=False)[0]['generated_text']
    # Strip the prompt from the output
    return output.replace(prompt, "").strip()

print("✅ Generator ready.")

In [13]:
%%writefile inference_engine.py
import paho.mqtt.client as mqtt
import json
import joblib
import numpy as np
from datetime import datetime
from sentence_transformers import SentenceTransformer
import faiss
from transformers import pipeline

# Load models
model = joblib.load("isolation_forest_v1.joblib")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
generator = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-135M-Instruct", device=-1)

# Load and index manual
with open("maintenance_manual.txt") as f:
    chunks = [c.strip() for c in f.read().split("\n\n") if c.strip()]
index = faiss.IndexFlatL2(embedder.encode(chunks).shape[1])
index.add(embedder.encode(chunks).astype('float32'))

def retrieve(query: str) -> str:
    vec = embedder.encode([query]).astype('float32')
    _, idx = index.search(vec, 1)
    return chunks[idx[0][0]]

def rag_diagnose(rms: float, kurtosis: float) -> str:
    query = f"vibration RMS {rms:.2f} kurtosis {kurtosis:.2f} bearing fault"
    context = retrieve(query)
    evidence = {"rms": rms, "kurtosis": kurtosis}
    
    prompt = f"Context: {context}\nEvidence: {evidence}\nGenerate maintenance recommendation:"
    return generator(prompt, max_new_tokens=150, do_sample=False)[0]['generated_text'].replace(prompt, "").strip()

# MQTT logic (same as before, but calling rag_diagnose instead of diagnose_fault)
# ... [rest of the inference engine logic] ...

In [14]:
%%writefile inference_engine.py
import paho.mqtt.client as mqtt
import json
import joblib
import numpy as np
from datetime import datetime
import time
from sentence_transformers import SentenceTransformer
import faiss
from transformers import pipeline

print("🧠 Loading models and knowledge base (this may take a minute)...")

# 1. Load the Anomaly Detection Model
model = joblib.load("isolation_forest_v1.joblib")

# 2. Load RAG Components
embedder = SentenceTransformer('all-MiniLM-L6-v2')
generator = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-135M-Instruct", device=-1)

with open("maintenance_manual.txt") as f:
    chunks = [c.strip() for c in f.read().split("\n\n") if c.strip()]

index = faiss.IndexFlatL2(embedder.encode(chunks).shape[1])
index.add(embedder.encode(chunks).astype('float32'))

anomaly_streak = 0
persistence_threshold = 3
current_state = "NORMAL"

def rag_diagnose(rms: float, kurtosis: float) -> tuple[str, float]:
    """Queries the knowledge base and generates a recommendation with latency tracking."""
    start_time = time.perf_counter()
    
    # Retrieve context
    query = f"vibration RMS {rms:.2f} kurtosis {kurtosis:.2f} bearing fault"
    query_vec = embedder.encode([query]).astype('float32')
    distances, indices = index.search(query_vec, 1)
    context = chunks[indices[0][0]]

    # Generate strict prompt
    prompt = f"""You are a maintenance copilot. Use ONLY the provided manual context to answer.
    Manual Context: {context}
    Sensor Evidence: RMS={rms:.3f}, Kurtosis={kurtosis:.3f}
    
    Provide:
    1. Fault Hypothesis
    2. Severity
    3. Recommended Actions
    4. Required Parts
    
    Recommendation:"""
    
    # Generate text
    output = generator(prompt, max_new_tokens=150, do_sample=False)[0]['generated_text']
    recommendation = output.replace(prompt, "").strip()
    
    latency = time.perf_counter() - start_time
    return recommendation, latency

def on_connect(client, userdata, flags, reason_code, properties):
    print("✅ Inference Engine Connected. Listening for anomalies...")
    client.subscribe("factory/motor_01/vibration")

def on_message(client, userdata, msg):
    global anomaly_streak, current_state
    payload = json.loads(msg.payload.decode())
    
    features = np.array([payload["features"]])
    rms = float(features[0][0])
    kurtosis = float(features[0][1])
    
    prediction = model.predict(features)[0]
    
    if prediction == -1:
        anomaly_streak += 1
    else:
        anomaly_streak = 0
        
    if anomaly_streak >= persistence_threshold:
        if current_state != "CRITICAL":
            print(f"\n🚨 ANOMALY CONFIRMED on {payload['sensor_id']}. Initiating RAG Copilot...")
            
            # Execute RAG Pipeline
            recommendation, latency = rag_diagnose(rms, kurtosis)
            
            # Build structured payload
            alert_payload = {
                "timestamp_utc": datetime.utcnow().isoformat() + "Z",
                "asset_id": payload["sensor_id"],
                "status": "CRITICAL",
                "evidence": {
                    "rms": round(rms, 3),
                    "kurtosis": round(kurtosis, 3)
                },
                "copilot_recommendation": recommendation,
                "telemetry": {
                    "rag_latency_seconds": round(latency, 3)
                }
            }
            
            # Output for dashboard/system
            print("\n--- 🚨 COPILOT DIAGNOSIS PAYLOAD ---")
            print(json.dumps(alert_payload, indent=4))
            print("-" * 50)
            
            current_state = "CRITICAL"
            
    elif anomaly_streak == 0 and current_state == "CRITICAL":
        print("\n✅ RECOVERY: Motor has returned to normal operation.")
        current_state = "NORMAL"

client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
client.on_connect = on_connect
client.on_message = on_message
client.connect("localhost", 1883, 60)
client.loop_forever()

In [15]:
import time

# 1. CLEAN SLATE
print("🧹 Cleaning up old processes...")
!pkill -f inference_engine.py || true
!pkill -f sensor_publisher.py || true
!pkill mosquitto || true
time.sleep(1)

# 2. START BROKER
print("🚀 Starting MQTT Broker...")
!apt-get install mosquitto mosquitto-clients -y > /dev/null
!mosquitto -d
time.sleep(1)

# 3. RUN PIPELINE
print("🧠 Starting RAG Inference Engine in the background...")
!rm -f inference_output.log
# Note: The first run might take 30-60 seconds to download the LLM
!python -u inference_engine.py > inference_output.log 2>&1 &
time.sleep(15) # Give the LLM time to load into memory before streaming starts

print("📡 Starting Sensor Publisher...")
!python sensor_publisher.py

print("\nWaiting for final messages to process...")
time.sleep(5)

print("\n--- 🚨 COPILOT RAG DIAGNOSIS LOGS ---")
!cat inference_output.log

In [16]:
%%writefile inference_engine.py
import paho.mqtt.client as mqtt
import json
import joblib
import numpy as np
from datetime import datetime
import time
from sentence_transformers import SentenceTransformer
import faiss
from transformers import pipeline

print("🧠 Loading models and knowledge base...", flush=True)

# 1. Load Anomaly Detection Model
model = joblib.load("isolation_forest_v1.joblib")

# 2. Load RAG Components
embedder = SentenceTransformer('all-MiniLM-L6-v2')
generator = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-135M-Instruct", device=-1)

with open("maintenance_manual.txt") as f:
    chunks = [c.strip() for c in f.read().split("\n\n") if c.strip()]

index = faiss.IndexFlatL2(embedder.encode(chunks).shape[1])
index.add(embedder.encode(chunks).astype('float32'))

anomaly_streak = 0
persistence_threshold = 3
current_state = "NORMAL"

def rag_diagnose(rms: float, kurtosis: float) -> tuple[str, float]:
    """Queries the knowledge base and generates a recommendation with latency tracking."""
    start_time = time.perf_counter()
    
    # Retrieve context
    query = f"vibration RMS {rms:.2f} kurtosis {kurtosis:.2f} bearing fault"
    query_vec = embedder.encode([query]).astype('float32')
    distances, indices = index.search(query_vec, 1)
    context = chunks[indices[0][0]]

    # Strict prompt
    prompt = f"""You are a maintenance copilot. Use ONLY the provided manual context to answer.
    Manual Context: {context}
    Sensor Evidence: RMS={rms:.3f}, Kurtosis={kurtosis:.3f}
    
    Provide:
    1. Fault Hypothesis
    2. Severity
    3. Recommended Actions
    4. Required Parts
    
    Recommendation:"""
    
    # Generate text (reduced tokens for speed)
    output = generator(prompt, max_new_tokens=100, do_sample=False)[0]['generated_text']
    recommendation = output.replace(prompt, "").strip()
    
    latency = time.perf_counter() - start_time
    return recommendation, latency

def on_connect(client, userdata, flags, reason_code, properties):
    print("✅ Inference Engine Connected. Listening for anomalies...", flush=True)
    client.subscribe("factory/motor_01/vibration")

def on_message(client, userdata, msg):
    global anomaly_streak, current_state
    payload = json.loads(msg.payload.decode())
    
    features = np.array([payload["features"]])
    rms = float(features[0][0])
    kurtosis = float(features[0][1])
    
    prediction = model.predict(features)[0]
    
    if prediction == -1:
        anomaly_streak += 1
    else:
        anomaly_streak = 0
        
    if anomaly_streak >= persistence_threshold:
        if current_state != "CRITICAL":
            print(f"\n🚨 ANOMALY CONFIRMED on {payload['sensor_id']}. Initiating RAG Copilot...", flush=True)
            
            # Execute RAG Pipeline
            recommendation, latency = rag_diagnose(rms, kurtosis)
            
            # Build structured payload
            alert_payload = {
                "timestamp_utc": datetime.utcnow().isoformat() + "Z",
                "asset_id": payload["sensor_id"],
                "status": "CRITICAL",
                "evidence": {
                    "rms": round(rms, 3),
                    "kurtosis": round(kurtosis, 3)
                },
                "copilot_recommendation": recommendation,
                "telemetry": {
                    "rag_latency_seconds": round(latency, 3)
                }
            }
            
            print("\n--- 🚨 COPILOT DIAGNOSIS PAYLOAD ---", flush=True)
            print(json.dumps(alert_payload, indent=4), flush=True)
            print("-" * 50, flush=True)
            
            current_state = "CRITICAL"
            
    elif anomaly_streak == 0 and current_state == "CRITICAL":
        print("\n✅ RECOVERY: Motor has returned to normal operation.", flush=True)
        current_state = "NORMAL"

client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
client.on_connect = on_connect
client.on_message = on_message
client.connect("localhost", 1883, 60)
client.loop_forever()

In [17]:
%%writefile inference_engine.py
import paho.mqtt.client as mqtt
import json
import joblib
import numpy as np
from datetime import datetime
import time
from sentence_transformers import SentenceTransformer
import faiss
from transformers import pipeline

print("🧠 Loading models and knowledge base...", flush=True)

# 1. Load Anomaly Detection Model
model = joblib.load("isolation_forest_v1.joblib")

# 2. Load RAG Components
embedder = SentenceTransformer('all-MiniLM-L6-v2')
generator = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-135M-Instruct", device=-1)

with open("maintenance_manual.txt") as f:
    chunks = [c.strip() for c in f.read().split("\n\n") if c.strip()]

index = faiss.IndexFlatL2(embedder.encode(chunks).shape[1])
index.add(embedder.encode(chunks).astype('float32'))

anomaly_streak = 0
persistence_threshold = 3
current_state = "NORMAL"

def rag_diagnose(rms: float, kurtosis: float) -> tuple[str, float]:
    """Queries the knowledge base and generates a recommendation with latency tracking."""
    start_time = time.perf_counter()
    
    # Retrieve context
    query = f"vibration RMS {rms:.2f} kurtosis {kurtosis:.2f} bearing fault"
    query_vec = embedder.encode([query]).astype('float32')
    distances, indices = index.search(query_vec, 1)
    context = chunks[indices[0][0]]

    # Strict prompt
    prompt = f"""You are a maintenance copilot. Use ONLY the provided manual context to answer.
    Manual Context: {context}
    Sensor Evidence: RMS={rms:.3f}, Kurtosis={kurtosis:.3f}
    
    Provide:
    1. Fault Hypothesis
    2. Severity
    3. Recommended Actions
    4. Required Parts
    
    Recommendation:"""
    
    # Generate text (reduced tokens for speed)
    output = generator(prompt, max_new_tokens=100, do_sample=False)[0]['generated_text']
    recommendation = output.replace(prompt, "").strip()
    
    latency = time.perf_counter() - start_time
    return recommendation, latency

def on_connect(client, userdata, flags, reason_code, properties):
    print("✅ Inference Engine Connected. Listening for anomalies...", flush=True)
    client.subscribe("factory/motor_01/vibration")

def on_message(client, userdata, msg):
    global anomaly_streak, current_state
    payload = json.loads(msg.payload.decode())
    
    features = np.array([payload["features"]])
    rms = float(features[0][0])
    kurtosis = float(features[0][1])
    
    prediction = model.predict(features)[0]
    
    if prediction == -1:
        anomaly_streak += 1
    else:
        anomaly_streak = 0
        
    if anomaly_streak >= persistence_threshold:
        if current_state != "CRITICAL":
            print(f"\n🚨 ANOMALY CONFIRMED on {payload['sensor_id']}. Initiating RAG Copilot...", flush=True)
            
            # Execute RAG Pipeline
            recommendation, latency = rag_diagnose(rms, kurtosis)
            
            # Build structured payload
            alert_payload = {
                "timestamp_utc": datetime.utcnow().isoformat() + "Z",
                "asset_id": payload["sensor_id"],
                "status": "CRITICAL",
                "evidence": {
                    "rms": round(rms, 3),
                    "kurtosis": round(kurtosis, 3)
                },
                "copilot_recommendation": recommendation,
                "telemetry": {
                    "rag_latency_seconds": round(latency, 3)
                }
            }
            
            print("\n--- 🚨 COPILOT DIAGNOSIS PAYLOAD ---", flush=True)
            print(json.dumps(alert_payload, indent=4), flush=True)
            print("-" * 50, flush=True)
            
            current_state = "CRITICAL"
            
    elif anomaly_streak == 0 and current_state == "CRITICAL":
        print("\n✅ RECOVERY: Motor has returned to normal operation.", flush=True)
        current_state = "NORMAL"

client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
client.on_connect = on_connect
client.on_message = on_message
client.connect("localhost", 1883, 60)
client.loop_forever()

In [18]:
import time

# 1. CLEAN SLATE
print("🧹 Cleaning up old processes...")
!pkill -f inference_engine.py || true
!pkill -f sensor_publisher.py || true
!pkill mosquitto || true
time.sleep(1)

# 2. START BROKER
print("🚀 Starting MQTT Broker...")
!apt-get install mosquitto mosquitto-clients -y > /dev/null
!mosquitto -d
time.sleep(1)

# 3. RUN PIPELINE
print("🧠 Starting RAG Inference Engine in the background...")
!rm -f inference_output.log
!python -u inference_engine.py > inference_output.log 2>&1 &
time.sleep(15) # Allow models to load

print("📡 Starting Sensor Publisher...")
!python sensor_publisher.py

print("\nWaiting for RAG generation to complete (20 seconds)...")
time.sleep(20) # Crucial: allow LLM generation to finish

# 4. CAPTURE AND CLEANUP
print("\n--- 🚨 COPILOT RAG DIAGNOSIS LOGS ---")
!cat inference_output.log

# Kill the background engine to free resources
!pkill -f inference_engine.py || true

In [19]:
import time
import json
from sentence_transformers import SentenceTransformer
import faiss
from transformers import pipeline

print("🧠 Loading Models...")
# Load RAG Components
embedder = SentenceTransformer('all-MiniLM-L6-v2')
generator = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-135M-Instruct", device=-1)

# Load Manual
with open("maintenance_manual.txt") as f:
    chunks = [c.strip() for c in f.read().split("\n\n") if c.strip()]

index = faiss.IndexFlatL2(embedder.encode(chunks).shape[1])
index.add(embedder.encode(chunks).astype('float32'))

def test_rag_diagnose(rms: float, kurtosis: float):
    print(f"\n🔍 Diagnosing Fault: RMS={rms}, Kurtosis={kurtosis}")
    start_time = time.perf_counter()
    
    # Retrieve
    query = f"vibration RMS {rms:.2f} kurtosis {kurtosis:.2f} bearing fault"
    query_vec = embedder.encode([query]).astype('float32')
    distances, indices = index.search(query_vec, 1)
    context = chunks[indices[0][0]]
    print(f"📄 Retrieved Context:\n{context[:150]}...\n")

    # Generate
    prompt = f"""You are a maintenance copilot. Use ONLY the provided manual context to answer.
    Manual Context: {context}
    Sensor Evidence: RMS={rms:.3f}, Kurtosis={kurtosis:.3f}
    
    Provide:
    1. Fault Hypothesis
    2. Severity
    3. Recommended Actions
    4. Required Parts
    
    Recommendation:"""
    
    print("⚙️ Generating Recommendation (Please wait ~10 seconds)...")
    output = generator(prompt, max_new_tokens=100, do_sample=False)[0]['generated_text']
    recommendation = output.replace(prompt, "").strip()
    
    latency = time.perf_counter() - start_time
    print(f"\n✅ Generated in {latency:.2f} seconds:\n")
    print("-" * 50)
    print(recommendation)
    print("-" * 50)

# TEST IT WITH THE DEGRADED VALUES
test_rag_diagnose(rms=0.293, kurtosis=5.1)

In [20]:
import time
from transformers import pipeline

print("🧠 Loading Qwen2.5-0.5B-Instruct (Please wait for download)...")
# Load a model that is actually smart enough to follow instructions
generator = pipeline(
    "text-generation", 
    model="Qwen/Qwen2.5-0.5B-Instruct", 
    device=-1 # CPU
)

# We hardcode the CORRECT context from the manual to isolate the LLM's generation ability
context = """
Fault: Outer-Race Bearing Degradation
Symptoms: High vibration RMS (>0.25g), elevated kurtosis (>4.0), spectral energy at BPFO frequency.
Immediate Action: Schedule vibration inspection within 24 hours.
Inspection Procedure: Remove bearing housing. Inspect outer race for spalling, pitting, or discoloration. Check lubrication for contamination (metal particles, water).
Corrective Action: Replace bearing with SKF-6205-2RS. Clean housing. Apply high-temp grease (Lithium complex, NLGI 2).
Estimated Downtime: 2.0 hours.
Required Parts: 1x Bearing SKF-6205-2RS, 50g High-temp grease.
"""

rms = 0.293
kurtosis = 5.1

# Proper Chat Template
messages = [
    {"role": "system", "content": "You are a maintenance copilot. You must answer using ONLY the provided manual context. Do not hallucinate."},
    {"role": "user", "content": f"Manual Context:\n{context}\n\nSensor Evidence: RMS={rms}, Kurtosis={kurtosis}.\n\nProvide a structured maintenance recommendation including Fault Hypothesis, Severity, Actions, and Parts."}
]

print("\n🔍 Generating Recommendation...")
start_time = time.perf_counter()

# Generate using the chat template
output = generator(messages, max_new_tokens=150, do_sample=False)
recommendation = output[0]['generated_text'][-1]['content']

latency = time.perf_counter() - start_time
print(f"✅ Generated in {latency:.2f} seconds:\n")
print("-" * 50)
print(recommendation)
print("-" * 50)

In [21]:
%%writefile inference_engine.py
import paho.mqtt.client as mqtt
import json
import joblib
import numpy as np
from datetime import datetime
import time
import threading
from groq import Groq

# --- 1. INITIALIZE CLIENTS ---
model = joblib.load("isolation_forest_v1.joblib")
groq_client = Groq(api_key="YOUR_GROQ_KEY_HERE") # <--- PASTE YOUR KEY HERE

anomaly_streak = 0
persistence_threshold = 3
current_state = "NORMAL"
is_generating = False # Lock to prevent multiple concurrent API calls

def generate_report_async(asset_id: str, rms: float, kurtosis: float):
    """Runs in a separate thread so the MQTT loop is not blocked."""
    global is_generating
    print(f"\n🧠 [BACKGROUND] Generating report for {asset_id}...", flush=True)
    
    start_time = time.perf_counter()
    
    # We skip FAISS for this test and inject the exact context directly
    context = """
    Fault: Outer-Race Bearing Degradation
    Symptoms: High vibration RMS (>0.25g), elevated kurtosis (>4.0).
    Immediate Action: Schedule vibration inspection within 24 hours.
    Inspection Procedure: Remove bearing housing. Inspect outer race for spalling. Check lubrication for contamination.
    Corrective Action: Replace bearing with SKF-6205-2RS. Apply high-temp grease.
    Estimated Downtime: 2.0 hours.
    Required Parts: 1x Bearing SKF-6205-2RS, 50g High-temp grease.
    """
    
    try:
        # Call Groq API (Llama-3-8B) - This takes ~300ms
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {"role": "system", "content": "You are a maintenance copilot. Use ONLY the provided context. Format output as JSON with keys: hypothesis, severity, actions (list), parts (list), downtime_hours."},
                {"role": "user", "content": f"Context:\n{context}\n\nEvidence: RMS={rms}, Kurtosis={kurtosis}"}
            ],
            model="llama3-8b-8192",
            temperature=0.1,
            max_tokens=500
        )
        
        recommendation = chat_completion.choices[0].message.content
        latency = time.perf_counter() - start_time
        
        # Build structured payload
        alert_payload = {
            "timestamp_utc": datetime.utcnow().isoformat() + "Z",
            "asset_id": asset_id,
            "status": "CRITICAL",
            "evidence": {"rms": round(rms, 3), "kurtosis": round(kurtosis, 3)},
            "copilot_recommendation": recommendation,
            "telemetry": {"api_latency_seconds": round(latency, 3)}
        }
        
        print("\n--- 🚨 COPILOT DIAGNOSIS PAYLOAD ---", flush=True)
        print(json.dumps(alert_payload, indent=4), flush=True)
        print("-" * 50, flush=True)
        
    except Exception as e:
        print(f"❌ [BACKGROUND] Groq API Error: {e}", flush=True)
    finally:
        is_generating = False

def on_connect(client, userdata, flags, reason_code, properties):
    print("✅ Inference Engine Connected. Listening...", flush=True)
    client.subscribe("factory/motor_01/vibration")

def on_message(client, userdata, msg):
    global anomaly_streak, current_state, is_generating
    payload = json.loads(msg.payload.decode())
    
    features = np.array([payload["features"]])
    rms = float(features[0][0])
    kurtosis = float(features[0][1])
    
    prediction = model.predict(features)[0]
    
    if prediction == -1:
        anomaly_streak += 1
    else:
        anomaly_streak = 0
        
    if anomaly_streak >= persistence_threshold:
        if current_state != "CRITICAL" and not is_generating:
            current_state = "CRITICAL"
            is_generating = True
            print(f"\n🚨 ANOMALY CONFIRMED on {payload['sensor_id']}. Spawning background thread...", flush=True)
            
            # Spawn background thread so MQTT keeps listening
            thread = threading.Thread(
                target=generate_report_async, 
                args=(payload["sensor_id"], rms, kurtosis)
            )
            thread.start()
            
    elif anomaly_streak == 0 and current_state == "CRITICAL":
        print("\n✅ RECOVERY: Motor has returned to normal operation.", flush=True)
        current_state = "NORMAL"

client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
client.on_connect = on_connect
client.on_message = on_message
client.connect("localhost", 1883, 60)
client.loop_forever()

In [22]:
import time

print("🧹 Cleaning up...")
!pkill -f inference_engine.py || true
!pkill -f sensor_publisher.py || true
!pkill mosquitto || true
time.sleep(1)

print("🚀 Starting MQTT Broker...")
!apt-get install mosquitto mosquitto-clients -y > /dev/null
!mosquitto -d
time.sleep(1)

print("🧠 Starting Inference Engine...")
!rm -f inference_output.log
!python -u inference_engine.py > inference_output.log 2>&1 &
time.sleep(3) # Give it time to connect

print("📡 Starting Sensor Publisher...")
!python sensor_publisher.py

print("\nWaiting for Background API to finish...")
time.sleep(5) # Groq is so fast, 5 seconds is plenty

print("\n--- 🚨 COPILOT RAG DIAGNOSIS LOGS ---")
!cat inference_output.log

!pkill -f inference_engine.py || true

In [23]:
# ==============================================================================
# 🚀 AEGISVISION / MAINTENANCE COPILOT: MONOLITHIC EXECUTION CELL
# ==============================================================================

# 1. INSTALL DEPENDENCIES (Silent)
!pip install paho-mqtt groq joblib scipy scikit-learn numpy -q

import time
import threading
import json
import subprocess
import numpy as np
import scipy.io as sio
from scipy.stats import kurtosis, skew
from sklearn.ensemble import IsolationForest
import joblib
import paho.mqtt.client as mqtt
from groq import Groq

# --- 2. CONFIGURATION ---
GROQ_API_KEY = "YOUR_GROQ_KEY_HERE" # <--- PASTE YOUR NEW KEY HERE
groq_client = Groq(api_key=GROQ_API_KEY)

# --- 3. PREPARE DATA & MODEL IN MEMORY ---
print("⏳ Preparing Data & Model...")
# Generate synthetic data
normal_signal = np.random.normal(0, 0.07, 50000)
faulty_signal = np.random.normal(0, 0.3, 50000)
r2f_timeline = np.concatenate([normal_signal, faulty_signal])
noisy_timeline = r2f_timeline + np.random.normal(0, 0.01, len(r2f_timeline))

# Train model
def extract_features(window):
    rms = np.sqrt(np.mean(window**2))
    return [rms, kurtosis(window), np.max(np.abs(window)) / rms if rms > 0 else 0, skew(window)]

X_train = np.array([extract_features(normal_signal[i:i+1024]) for i in range(0, 40000, 512)])
model = IsolationForest(contamination=0.01, random_state=42)
model.fit(X_train)
print("✅ Model Ready.")

# --- 4. START MOSQUITTO BROKER ---
print("🚀 Starting MQTT Broker...")
subprocess.run(["apt-get", "install", "mosquitto", "mosquitto-clients", "-y"], stdout=subprocess.DEVNULL)
subprocess.run(["pkill", "mosquitto"])
subprocess.run(["mosquitto", "-d"])
time.sleep(1)

# --- 5. DEFINE SUBSCRIBER (GROQ API) ---
anomaly_streak = 0
current_state = "NORMAL"

def on_message(client, userdata, msg):
    global anomaly_streak, current_state
    payload = json.loads(msg.payload.decode())
    features = np.array([payload["features"]])
    rms = float(features[0][0])
    kurtosis = float(features[0][1])
    
    prediction = model.predict(features)[0]
    
    if prediction == -1:
        anomaly_streak += 1
    else:
        anomaly_streak = 0
        
    if anomaly_streak >= 3:
        if current_state != "CRITICAL":
            current_state = "CRITICAL"
            print(f"\n🚨 ANOMALY CONFIRMED on {payload['sensor_id']}. Calling Groq API...")
            
            start_time = time.perf_counter()
            try:
                completion = groq_client.chat.completions.create(
                    messages=[
                        {"role": "system", "content": "You are a maintenance copilot. Output strict JSON with keys: hypothesis, severity, actions, parts, downtime."},
                        {"role": "user", "content": f"Context: Outer-race bearing degradation. Evidence: RMS={rms:.3f}, Kurtosis={kurtosis:.3f}"}
                    ],
                    model="llama3-8b-8192",
                    temperature=0.1,
                    max_tokens=300
                )
                latency = time.perf_counter() - start_time
                print(f"✅ Groq Response ({latency:.2f}s):")
                print(completion.choices[0].message.content)
                print("-" * 50)
            except Exception as e:
                print(f"❌ Groq Error: {e}")

def run_subscriber():
    client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
    client.on_message = on_message
    client.connect("localhost", 1883, 60)
    client.subscribe("factory/motor_01/vibration")
    client.loop_forever()

# Start Subscriber in Background Thread
threading.Thread(target=run_subscriber, daemon=True).start()
time.sleep(1)

# --- 6. PUBLISHER (RUNS IN MAIN THREAD) ---
print("📡 Starting Sensor Publisher...")
pub_client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
pub_client.connect("localhost", 1883, 60)

window_size = 1024
step_size = 512

for i in range(0, len(noisy_timeline) - window_size, step_size):
    window = noisy_timeline[i : i + window_size]
    rms = float(np.sqrt(np.mean(window**2)))
    kurt = float(kurtosis(window))
    crest = float(np.max(np.abs(window)) / rms if rms > 0 else 0)
    skew_val = float(skew(window))
    
    payload = {"sensor_id": "MOTOR-01", "features": [rms, kurt, crest, skew_val]}
    pub_client.publish("factory/motor_01/vibration", json.dumps(payload))
    
    if i % (step_size * 20) == 0:
        print(f"Streaming... Sample {i}/{len(noisy_timeline)} | RMS: {rms:.3f}")
    time.sleep(0.01) # Fast stream

print("\n✅ Pipeline Execution Complete.")
time.sleep(3) # Give Groq API time to finish and print

In [24]:
# ==============================================================================
# 🚀 MONOLITHIC EXECUTION CELL V2 (Cloud Broker + Async Groq)
# ==============================================================================
import time
import threading
import json
import numpy as np
from scipy.stats import kurtosis, skew
from sklearn.ensemble import IsolationForest
import paho.mqtt.client as mqtt
from groq import Groq

# --- 1. CONFIGURATION ---
GROQ_API_KEY = "YOUR_GROQ_KEY_HERE" # <--- PASTE YOUR KEY HERE
# Using a free public broker to avoid local Colab daemon issues
BROKER_ADDRESS = "broker.hivemq.com" 
PORT = 1883
TOPIC = "aegisvision/factory/motor_01"

groq_client = Groq(api_key=GROQ_API_KEY)

# --- 2. PREPARE DATA & MODEL IN MEMORY ---
print("⏳ Preparing Data & Model...")
# Generate synthetic data
normal_signal = np.random.normal(0, 0.07, 50000)
faulty_signal = np.random.normal(0, 0.3, 50000)
r2f_timeline = np.concatenate([normal_signal, faulty_signal])
noisy_timeline = r2f_timeline + np.random.normal(0, 0.01, len(r2f_timeline))

def extract_features(window):
    rms = np.sqrt(np.mean(window**2))
    return [rms, kurtosis(window), np.max(np.abs(window)) / rms if rms > 0 else 0, skew(window)]

X_train = np.array([extract_features(normal_signal[i:i+1024]) for i in range(0, 40000, 512)])
model = IsolationForest(contamination=0.01, random_state=42)
model.fit(X_train)
print("✅ Model Ready.")

# --- 3. DEFINE SUBSCRIBER (GROQ API) ---
anomaly_streak = 0
current_state = "NORMAL"

def on_connect(client, userdata, flags, rc, properties):
    print(f"🧠 Subscriber connected to {BROKER_ADDRESS}. Listening...")
    client.subscribe(TOPIC)

def on_message(client, userdata, msg):
    global anomaly_streak, current_state
    payload = json.loads(msg.payload.decode())
    features = np.array([payload["features"]])
    rms = float(features[0][0])
    kurtosis_val = float(features[0][1])
    
    prediction = model.predict(features)[0]
    
    if prediction == -1:
        anomaly_streak += 1
    else:
        anomaly_streak = 0
        
    if anomaly_streak >= 3:
        if current_state != "CRITICAL":
            current_state = "CRITICAL"
            print(f"\n🚨 ANOMALY CONFIRMED! (RMS: {rms:.3f}). Calling Groq API...")
            
            start_time = time.perf_counter()
            try:
                completion = groq_client.chat.completions.create(
                    messages=[
                        {"role": "system", "content": "You are a maintenance copilot. Output strict JSON with keys: hypothesis, severity, actions, parts, downtime."},
                        {"role": "user", "content": f"Context: Outer-race bearing degradation. Evidence: RMS={rms:.3f}, Kurtosis={kurtosis_val:.3f}"}
                    ],
                    model="llama3-8b-8192",
                    temperature=0.1,
                    max_tokens=300
                )
                latency = time.perf_counter() - start_time
                print(f"✅ Groq Response ({latency:.2f}s):")
                print(completion.choices[0].message.content)
                print("-" * 50)
            except Exception as e:
                print(f"❌ Groq Error: {e}")

def run_subscriber():
    client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
    client.on_connect = on_connect
    client.on_message = on_message
    client.connect(BROKER_ADDRESS, PORT, 60)
    client.loop_forever()

# Start Subscriber in Background Thread
threading.Thread(target=run_subscriber, daemon=True).start()
time.sleep(2) # Wait for connection

# --- 4. PUBLISHER (RUNS IN MAIN THREAD) ---
print(f"📡 Starting Sensor Publisher to {BROKER_ADDRESS}...")
pub_client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
pub_client.connect(BROKER_ADDRESS, PORT, 60)

window_size = 1024
step_size = 512

for i in range(0, len(noisy_timeline) - window_size, step_size):
    window = noisy_timeline[i : i + window_size]
    rms = float(np.sqrt(np.mean(window**2)))
    kurt = float(kurtosis(window))
    crest = float(np.max(np.abs(window)) / rms if rms > 0 else 0)
    skew_val = float(skew(window))
    
    payload = {"sensor_id": "MOTOR-01", "features": [rms, kurt, crest, skew_val]}
    pub_client.publish(TOPIC, json.dumps(payload))
    
    if i % (step_size * 20) == 0:
        print(f"Streaming... Sample {i}/{len(noisy_timeline)} | RMS: {rms:.3f}")
    time.sleep(0.01) 

print("\n✅ Pipeline Execution Complete. Waiting for API response...")
time.sleep(5) # Give Groq API time to finish and print

In [25]:
# ==============================================================================
# 🚀 MONOLITHIC EXECUTION CELL V3 (Bulletproof Threading)
# ==============================================================================
import time
import threading
import json
import numpy as np
from scipy.stats import kurtosis, skew
from sklearn.ensemble import IsolationForest
import paho.mqtt.client as mqtt
from groq import Groq

# --- 1. CONFIGURATION ---
# PASTE YOUR *NEW* KEY HERE (DO NOT SHARE IT)
GROQ_API_KEY = "YOUR_GROQ_KEY_HERE" 
BROKER_ADDRESS = "broker.hivemq.com" 
PORT = 1883
TOPIC = "aegisvision/factory/motor_01"

groq_client = Groq(api_key=GROQ_API_KEY)

# --- 2. PREPARE DATA & MODEL ---
print("⏳ Preparing Data & Model...")
normal_signal = np.random.normal(0, 0.07, 50000)
faulty_signal = np.random.normal(0, 0.3, 50000)
r2f_timeline = np.concatenate([normal_signal, faulty_signal])
noisy_timeline = r2f_timeline + np.random.normal(0, 0.01, len(r2f_timeline))

def extract_features(window):
    rms = np.sqrt(np.mean(window**2))
    return [rms, kurtosis(window), np.max(np.abs(window)) / rms if rms > 0 else 0, skew(window)]

X_train = np.array([extract_features(normal_signal[i:i+1024]) for i in range(0, 40000, 512)])
model = IsolationForest(contamination=0.01, random_state=42)
model.fit(X_train)
print("✅ Model Ready.")

# --- 3. DEFINE SUBSCRIBER & BACKGROUND WORKER ---
anomaly_streak = 0
current_state = "NORMAL"
bg_thread = None # Keep track of the thread

def call_groq_api(rms, kurtosis_val):
    print(f"\n🧠 [BACKGROUND] Calling Groq API for RMS={rms:.3f}...", flush=True)
    start_time = time.perf_counter()
    try:
        completion = groq_client.chat.completions.create(
            messages=[
                {"role": "system", "content": "You are a maintenance copilot. Output strict JSON with keys: hypothesis, severity, actions, parts, downtime."},
                {"role": "user", "content": f"Context: Outer-race bearing degradation. Evidence: RMS={rms:.3f}, Kurtosis={kurtosis_val:.3f}"}
            ],
            model="llama3-8b-8192",
            temperature=0.1,
            max_tokens=300
        )
        latency = time.perf_counter() - start_time
        print(f"✅ [BACKGROUND] Groq Response ({latency:.2f}s):", flush=True)
        print(completion.choices[0].message.content, flush=True)
        print("-" * 50, flush=True)
    except Exception as e:
        print(f"❌ [BACKGROUND] Groq API Error: {e}", flush=True)

def on_connect(client, userdata, flags, rc, properties):
    print(f"🧠 Subscriber connected to {BROKER_ADDRESS}. Listening...", flush=True)
    client.subscribe(TOPIC)

def on_message(client, userdata, msg):
    global anomaly_streak, current_state, bg_thread
    payload = json.loads(msg.payload.decode())
    features = np.array([payload["features"]])
    rms = float(features[0][0])
    kurtosis_val = float(features[0][1])
    
    prediction = model.predict(features)[0]
    
    if prediction == -1:
        anomaly_streak += 1
    else:
        anomaly_streak = 0
        
    if anomaly_streak >= 3:
        if current_state != "CRITICAL":
            current_state = "CRITICAL"
            print(f"\n🚨 ANOMALY CONFIRMED on MOTOR-01! Spawning Groq Thread...", flush=True)
            bg_thread = threading.Thread(target=call_groq_api, args=(rms, kurtosis_val))
            bg_thread.start()

def run_subscriber():
    client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
    client.on_connect = on_connect
    client.on_message = on_message
    client.connect(BROKER_ADDRESS, PORT, 60)
    client.loop_forever()

# Start Subscriber in Background Thread
threading.Thread(target=run_subscriber, daemon=True).start()
time.sleep(2)

# --- 4. PUBLISHER (RUNS IN MAIN THREAD) ---
print(f"📡 Starting Sensor Publisher to {BROKER_ADDRESS}...")
pub_client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
pub_client.connect(BROKER_ADDRESS, PORT, 60)

window_size = 1024
step_size = 512

for i in range(0, len(noisy_timeline) - window_size, step_size):
    window = noisy_timeline[i : i + window_size]
    rms = float(np.sqrt(np.mean(window**2)))
    kurt = float(kurtosis(window))
    crest = float(np.max(np.abs(window)) / rms if rms > 0 else 0)
    skew_val = float(skew(window))
    
    payload = {"sensor_id": "MOTOR-01", "features": [rms, kurt, crest, skew_val]}
    pub_client.publish(TOPIC, json.dumps(payload))
    
    if i % (step_size * 20) == 0:
        print(f"Streaming... Sample {i}/{len(noisy_timeline)} | RMS: {rms:.3f}")
    time.sleep(0.01) 

print("\n✅ Pipeline Execution Complete.")
print("⏳ Waiting for Groq API Thread to finish...")

# CRITICAL FIX: Force the main script to wait for the API call to finish
if bg_thread is not None:
    bg_thread.join(timeout=15) # Wait up to 15 seconds
else:
    print("⚠️ Warning: No anomaly triggered, so no API call was made.")

print("🏁 Execution Finished.")

In [26]:
# ==============================================================================
# 🚀 MONOLITHIC EXECUTION CELL V4 (Fixed Timing & Thread Sync)
# ==============================================================================
import time
import threading
import json
import numpy as np
from scipy.stats import kurtosis, skew
from sklearn.ensemble import IsolationForest
import paho.mqtt.client as mqtt
from groq import Groq

# --- 1. CONFIGURATION ---
GROQ_API_KEY = "YOUR_NEW_GROQ_KEY" # <--- PASTE YOUR KEY
BROKER_ADDRESS = "broker.hivemq.com" 
PORT = 1883
TOPIC = "aegisvision/factory/motor_01"

groq_client = Groq(api_key=GROQ_API_KEY)

# --- 2. PREPARE DATA & MODEL ---
print("⏳ Preparing Data & Model...")
normal_signal = np.random.normal(0, 0.07, 50000)
faulty_signal = np.random.normal(0, 0.3, 50000)
r2f_timeline = np.concatenate([normal_signal, faulty_signal])
noisy_timeline = r2f_timeline + np.random.normal(0, 0.01, len(r2f_timeline))

def extract_features(window):
    rms = np.sqrt(np.mean(window**2))
    return [rms, kurtosis(window), np.max(np.abs(window)) / rms if rms > 0 else 0, skew(window)]

X_train = np.array([extract_features(normal_signal[i:i+1024]) for i in range(0, 40000, 512)])
model = IsolationForest(contamination=0.01, random_state=42)
model.fit(X_train)
print("✅ Model Ready.")

# --- 3. DEFINE SUBSCRIBER ---
anomaly_streak = 0
current_state = "NORMAL"
bg_thread = None

def call_groq_api(rms, kurtosis_val):
    print(f"\n🧠 [BACKGROUND] Calling Groq API for RMS={rms:.3f}...", flush=True)
    start_time = time.perf_counter()
    try:
        completion = groq_client.chat.completions.create(
            messages=[
                {"role": "system", "content": "You are a maintenance copilot. Output strict JSON with keys: hypothesis, severity, actions, parts, downtime."},
                {"role": "user", "content": f"Context: Outer-race bearing degradation. Evidence: RMS={rms:.3f}, Kurtosis={kurtosis_val:.3f}"}
            ],
            model="llama3-8b-8192",
            temperature=0.1,
            max_tokens=300
        )
        latency = time.perf_counter() - start_time
        print(f"✅ [BACKGROUND] Groq Response ({latency:.2f}s):", flush=True)
        print(completion.choices[0].message.content, flush=True)
        print("-" * 50, flush=True)
    except Exception as e:
        print(f"❌ [BACKGROUND] Groq API Error: {e}", flush=True)

def on_connect(client, userdata, flags, rc, properties):
    print(f"🧠 Subscriber connected to {BROKER_ADDRESS}. Listening...", flush=True)
    client.subscribe(TOPIC)

def on_message(client, userdata, msg):
    global anomaly_streak, current_state, bg_thread
    payload = json.loads(msg.payload.decode())
    features = np.array([payload["features"]])
    rms = float(features[0][0])
    kurtosis_val = float(features[0][1])
    
    prediction = model.predict(features)[0]
    
    # DEBUG PRINT: Prove the subscriber is receiving messages
    if prediction == -1:
        print(f"  [Subscriber] Anomaly detected! RMS={rms:.3f}", flush=True)
        anomaly_streak += 1
    else:
        anomaly_streak = 0
        
    if anomaly_streak >= 3:
        if current_state != "CRITICAL":
            current_state = "CRITICAL"
            print(f"\n🚨 ANOMALY CONFIRMED on MOTOR-01! Spawning Groq Thread...", flush=True)
            bg_thread = threading.Thread(target=call_groq_api, args=(rms, kurtosis_val))
            bg_thread.start()

def run_subscriber():
    client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
    client.on_connect = on_connect
    client.on_message = on_message
    client.connect(BROKER_ADDRESS, PORT, 60)
    client.loop_forever()

# Start Subscriber in Background Thread
threading.Thread(target=run_subscriber, daemon=True).start()
time.sleep(2)

# --- 4. PUBLISHER ---
print(f"📡 Starting Sensor Publisher to {BROKER_ADDRESS}...")
pub_client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
pub_client.connect(BROKER_ADDRESS, PORT, 60)

window_size = 1024
step_size = 512

for i in range(0, len(noisy_timeline) - window_size, step_size):
    window = noisy_timeline[i : i + window_size]
    rms = float(np.sqrt(np.mean(window**2)))
    kurt = float(kurtosis(window))
    crest = float(np.max(np.abs(window)) / rms if rms > 0 else 0)
    skew_val = float(skew(window))
    
    payload = {"sensor_id": "MOTOR-01", "features": [rms, kurt, crest, skew_val]}
    pub_client.publish(TOPIC, json.dumps(payload))
    
    # Slowed down to simulate real network latency
    if i % (step_size * 20) == 0:
        print(f"Streaming... Sample {i}/{len(noisy_timeline)} | RMS: {rms:.3f}")
    time.sleep(0.05) # 50ms per message = ~10 seconds for the whole stream

print("\n✅ Publisher finished streaming.")

# --- 5. THE CRITICAL FIX: WAIT FOR THE SUBSCRIBER ---
print("⏳ Waiting 10 seconds for subscriber to process the queue and call Groq...")
time.sleep(10) 

if bg_thread is not None:
    print("⏳ Waiting for Groq API Thread to finish...")
    bg_thread.join(timeout=15) 

print("🏁 Execution Finished.")

In [27]:
# ==============================================================================
# 🚀 MONOLITHIC EXECUTION CELL V4 (Fixed Timing & Thread Sync)
# ==============================================================================
import time
import threading
import json
import numpy as np
from scipy.stats import kurtosis, skew
from sklearn.ensemble import IsolationForest
import paho.mqtt.client as mqtt
from groq import Groq

# --- 1. CONFIGURATION ---
GROQ_API_KEY = "YOUR_GROQ_KEY_HERE" # <--- PASTE YOUR KEY
BROKER_ADDRESS = "broker.hivemq.com" 
PORT = 1883
TOPIC = "aegisvision/factory/motor_01"

groq_client = Groq(api_key=GROQ_API_KEY)

# --- 2. PREPARE DATA & MODEL ---
print("⏳ Preparing Data & Model...")
normal_signal = np.random.normal(0, 0.07, 50000)
faulty_signal = np.random.normal(0, 0.3, 50000)
r2f_timeline = np.concatenate([normal_signal, faulty_signal])
noisy_timeline = r2f_timeline + np.random.normal(0, 0.01, len(r2f_timeline))

def extract_features(window):
    rms = np.sqrt(np.mean(window**2))
    return [rms, kurtosis(window), np.max(np.abs(window)) / rms if rms > 0 else 0, skew(window)]

X_train = np.array([extract_features(normal_signal[i:i+1024]) for i in range(0, 40000, 512)])
model = IsolationForest(contamination=0.01, random_state=42)
model.fit(X_train)
print("✅ Model Ready.")

# --- 3. DEFINE SUBSCRIBER ---
anomaly_streak = 0
current_state = "NORMAL"
bg_thread = None

def call_groq_api(rms, kurtosis_val):
    print(f"\n🧠 [BACKGROUND] Calling Groq API for RMS={rms:.3f}...", flush=True)
    start_time = time.perf_counter()
    try:
        completion = groq_client.chat.completions.create(
            messages=[
                {"role": "system", "content": "You are a maintenance copilot. Output strict JSON with keys: hypothesis, severity, actions, parts, downtime."},
                {"role": "user", "content": f"Context: Outer-race bearing degradation. Evidence: RMS={rms:.3f}, Kurtosis={kurtosis_val:.3f}"}
            ],
            model="llama3-8b-8192",
            temperature=0.1,
            max_tokens=300
        )
        latency = time.perf_counter() - start_time
        print(f"✅ [BACKGROUND] Groq Response ({latency:.2f}s):", flush=True)
        print(completion.choices[0].message.content, flush=True)
        print("-" * 50, flush=True)
    except Exception as e:
        print(f"❌ [BACKGROUND] Groq API Error: {e}", flush=True)

def on_connect(client, userdata, flags, rc, properties):
    print(f"🧠 Subscriber connected to {BROKER_ADDRESS}. Listening...", flush=True)
    client.subscribe(TOPIC)

def on_message(client, userdata, msg):
    global anomaly_streak, current_state, bg_thread
    payload = json.loads(msg.payload.decode())
    features = np.array([payload["features"]])
    rms = float(features[0][0])
    kurtosis_val = float(features[0][1])
    
    prediction = model.predict(features)[0]
    
    # DEBUG PRINT: Prove the subscriber is receiving messages
    if prediction == -1:
        print(f"  [Subscriber] Anomaly detected! RMS={rms:.3f}", flush=True)
        anomaly_streak += 1
    else:
        anomaly_streak = 0
        
    if anomaly_streak >= 3:
        if current_state != "CRITICAL":
            current_state = "CRITICAL"
            print(f"\n🚨 ANOMALY CONFIRMED on MOTOR-01! Spawning Groq Thread...", flush=True)
            bg_thread = threading.Thread(target=call_groq_api, args=(rms, kurtosis_val))
            bg_thread.start()

def run_subscriber():
    client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
    client.on_connect = on_connect
    client.on_message = on_message
    client.connect(BROKER_ADDRESS, PORT, 60)
    client.loop_forever()

# Start Subscriber in Background Thread
threading.Thread(target=run_subscriber, daemon=True).start()
time.sleep(2)

# --- 4. PUBLISHER ---
print(f"📡 Starting Sensor Publisher to {BROKER_ADDRESS}...")
pub_client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
pub_client.connect(BROKER_ADDRESS, PORT, 60)

window_size = 1024
step_size = 512

for i in range(0, len(noisy_timeline) - window_size, step_size):
    window = noisy_timeline[i : i + window_size]
    rms = float(np.sqrt(np.mean(window**2)))
    kurt = float(kurtosis(window))
    crest = float(np.max(np.abs(window)) / rms if rms > 0 else 0)
    skew_val = float(skew(window))
    
    payload = {"sensor_id": "MOTOR-01", "features": [rms, kurt, crest, skew_val]}
    pub_client.publish(TOPIC, json.dumps(payload))
    
    # Slowed down to simulate real network latency
    if i % (step_size * 20) == 0:
        print(f"Streaming... Sample {i}/{len(noisy_timeline)} | RMS: {rms:.3f}")
    time.sleep(0.05) # 50ms per message = ~10 seconds for the whole stream

print("\n✅ Publisher finished streaming.")

# --- 5. THE CRITICAL FIX: WAIT FOR THE SUBSCRIBER ---
print("⏳ Waiting 10 seconds for subscriber to process the queue and call Groq...")
time.sleep(10) 

if bg_thread is not None:
    print("⏳ Waiting for Groq API Thread to finish...")
    bg_thread.join(timeout=15) 

print("🏁 Execution Finished.")

In [28]:
# ==============================================================================
# 🚀 MONOLITHIC EXECUTION CELL V5 (Groq Model Fix + Hardened Detector)
# ==============================================================================
import os, time, threading, json
import numpy as np
from scipy.stats import kurtosis, skew
from sklearn.ensemble import IsolationForest
import paho.mqtt.client as mqtt
from groq import Groq

# --- 1. SECURE API KEY ---
# Set this in Colab Secrets (recommended) or paste your key below.
# NEVER hardcode it in a public notebook.
GROQ_API_KEY = "YOUR_GROQ_KEY_HERE"  # <--- Replace with your new key
BROKER_ADDRESS = "broker.hivemq.com"
PORT = 1883
TOPIC = "aegisvision/factory/motor_01"

groq_client = Groq(api_key=GROQ_API_KEY)

# --- 2. TRAIN A ROBUST ISOLATION FOREST ---
print("⏳ Training robust Isolation Forest...")
# Use a wider standard deviation for 'normal' training data to prevent overfitting
normal_train = np.random.normal(0, 0.12, 50000)  # 0.12 instead of 0.07
faulty_signal = np.random.normal(0, 0.3, 50000)
timeline = np.concatenate([normal_train, faulty_signal]) + np.random.normal(0, 0.01, 100000)

def extract_features(window):
    rms = np.sqrt(np.mean(window**2))
    return [rms, kurtosis(window), np.max(np.abs(window)) / rms if rms > 0 else 0, skew(window)]

X_train = np.array([extract_features(normal_train[i:i+1024]) for i in range(0, 40000, 512)])
# Raise contamination from 0.01 to 0.05 to allow for normal variance
model = IsolationForest(contamination=0.05, random_state=42).fit(X_train)
print("✅ Model trained. Anomaly threshold is now more conservative.")

# --- 3. SUBSCRIBER & BACKGROUND THREAD ---
anomaly_streak = 0
current_state = "NORMAL"
bg_thread = None

def call_groq_api(rms, kurtosis_val):
    print(f"\n🧠 [BACKGROUND] Calling Groq API for RMS={rms:.3f}...", flush=True)
    start_time = time.perf_counter()
    try:
        completion = groq_client.chat.completions.create(
            messages=[
                {"role": "system", "content": "You are a maintenance copilot. Output strict JSON with keys: hypothesis, severity, actions, parts, downtime."},
                {"role": "user", "content": f"Context: Outer-race bearing degradation. Evidence: RMS={rms:.3f}, Kurtosis={kurtosis_val:.3f}"}
            ],
            # ✅ UPDATED MODEL ID: llama3-8b-8192 is decommissioned.
            # Use "openai/gpt-oss-20b" or "llama-3.3-70b-versatile".
            model="openai/gpt-oss-20b",
            temperature=0.1,
            max_tokens=300
        )
        latency = time.perf_counter() - start_time
        print(f"✅ [BACKGROUND] Groq Response ({latency:.2f}s):", flush=True)
        print(completion.choices[0].message.content, flush=True)
        print("-" * 50, flush=True)
    except Exception as e:
        print(f"❌ [BACKGROUND] Groq API Error: {e}", flush=True)

def on_connect(client, userdata, flags, rc, properties):
    print(f"🧠 Subscriber connected. Listening...", flush=True)
    client.subscribe(TOPIC)

def on_message(client, userdata, msg):
    global anomaly_streak, current_state, bg_thread
    payload = json.loads(msg.payload.decode())
    features = np.array([payload["features"]])
    rms = float(features[0][0])
    kurtosis_val = float(features[0][1])

    prediction = model.predict(features)[0]

    if prediction == -1:
        print(f"  [Subscriber] Anomaly detected! RMS={rms:.3f}", flush=True)
        anomaly_streak += 1
    else:
        anomaly_streak = 0

    if anomaly_streak >= 3 and current_state != "CRITICAL":
        current_state = "CRITICAL"
        print(f"\n🚨 ANOMALY CONFIRMED on MOTOR-01! Spawning Groq Thread...", flush=True)
        bg_thread = threading.Thread(target=call_groq_api, args=(rms, kurtosis_val))
        bg_thread.start()

def run_subscriber():
    client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
    client.on_connect = on_connect
    client.on_message = on_message
    client.connect(BROKER_ADDRESS, PORT, 60)
    client.loop_forever()

threading.Thread(target=run_subscriber, daemon=True).start()
time.sleep(2)

# --- 4. PUBLISHER (SIMULATED SENSOR) ---
print(f"📡 Streaming data to {BROKER_ADDRESS}...")
pub_client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
pub_client.connect(BROKER_ADDRESS, PORT, 60)

window_size, step_size = 1024, 512
for i in range(0, len(timeline) - window_size, step_size):
    window = timeline[i:i+window_size]
    rms = float(np.sqrt(np.mean(window**2)))
    kurt = float(kurtosis(window))
    crest = float(np.max(np.abs(window)) / rms if rms > 0 else 0)
    skew_val = float(skew(window))
    payload = {"sensor_id": "MOTOR-01", "features": [rms, kurt, crest, skew_val]}
    pub_client.publish(TOPIC, json.dumps(payload))
    if i % (step_size * 20) == 0:
        print(f"Streaming... RMS: {rms:.3f}")
    time.sleep(0.05)  # Simulate real-time stream

print("\n✅ Publisher finished.")
print("⏳ Waiting for Groq API Thread to finish...")

if bg_thread is not None:
    bg_thread.join(timeout=15)
else:
    print("⚠️ No anomaly was confirmed by the detector.")

print("🏁 Execution Finished.")

In [29]:
# ==============================================================================
# 🚀 MONOLITHIC EXECUTION CELL V6 (Hybrid Rules + Strict JSON)
# ==============================================================================
import time, threading, json
import numpy as np
from scipy.stats import kurtosis, skew
from sklearn.ensemble import IsolationForest
import paho.mqtt.client as mqtt
from groq import Groq

# --- 1. CONFIGURATION ---
GROQ_API_KEY = "YOUR_GROQ_KEY_HERE" # <--- PASTE YOUR KEY
BROKER_ADDRESS = "broker.hivemq.com" 
PORT = 1883
TOPIC = "aegisvision/factory/motor_01"

groq_client = Groq(api_key=GROQ_API_KEY)

# --- 2. PREPARE DATA & MODEL ---
print("⏳ Training Isolation Forest...")
normal_train = np.random.normal(0, 0.12, 50000)
faulty_signal = np.random.normal(0, 0.3, 50000)
timeline = np.concatenate([normal_train, faulty_signal]) + np.random.normal(0, 0.01, 100000)

def extract_features(window):
    rms = np.sqrt(np.mean(window**2))
    return [rms, kurtosis(window), np.max(np.abs(window)) / rms if rms > 0 else 0, skew(window)]

X_train = np.array([extract_features(normal_train[i:i+1024]) for i in range(0, 40000, 512)])
model = IsolationForest(contamination=0.01, random_state=42).fit(X_train)
print("✅ Model trained.")

# --- 3. SUBSCRIBER & BACKGROUND WORKER ---
anomaly_streak = 0
current_state = "NORMAL"
bg_thread = None

def call_groq_api(rms, kurtosis_val):
    print(f"\n🧠 [BACKGROUND] Calling Groq API (JSON Mode) for RMS={rms:.3f}...", flush=True)
    start_time = time.perf_counter()
    try:
        # We force JSON mode and give a strict schema in the prompt
        completion = groq_client.chat.completions.create(
            messages=[
                {"role": "system", "content": "You are a maintenance copilot. Output ONLY valid JSON with keys: hypothesis (string), severity (string), actions (list of strings), parts (list of strings), downtime (string)."},
                {"role": "user", "content": f"Context: Outer-race bearing degradation. Evidence: RMS={rms:.3f}, Kurtosis={kurtosis_val:.3f}"}
            ],
            model="openai/gpt-oss-20b",
            temperature=0.1,
            max_tokens=400,
            response_format={"type": "json_object"} # <--- CRITICAL FIX: Forces valid JSON
        )
        latency = time.perf_counter() - start_time
        response_json = completion.choices[0].message.content
        
        print(f"✅ [BACKGROUND] Groq Response ({latency:.2f}s):", flush=True)
        print(response_json, flush=True)
        print("-" * 50, flush=True)
    except Exception as e:
        print(f"❌ [BACKGROUND] Groq API Error: {e}", flush=True)

def on_connect(client, userdata, flags, rc, properties):
    print(f"🧠 Subscriber connected. Listening...", flush=True)
    client.subscribe(TOPIC)

def on_message(client, userdata, msg):
    global anomaly_streak, current_state, bg_thread
    payload = json.loads(msg.payload.decode())
    features = np.array([payload["features"]])
    rms = float(features[0][0])
    kurtosis_val = float(features[0][1])

    prediction = model.predict(features)[0]

    # HYBRID RULE: The ML model must say it's an anomaly AND the RMS must be > 0.20
    # This prevents the model from triggering on normal variance (RMS ~0.115)
    is_real_anomaly = (prediction == -1) and (rms > 0.20)

    if is_real_anomaly:
        print(f"  [Subscriber] REAL Anomaly detected! RMS={rms:.3f}", flush=True)
        anomaly_streak += 1
    else:
        anomaly_streak = 0 # Reset streak if it's normal or just noise

    if anomaly_streak >= 3 and current_state != "CRITICAL":
        current_state = "CRITICAL"
        print(f"\n🚨 ANOMALY CONFIRMED on MOTOR-01! Spawning Groq Thread...", flush=True)
        bg_thread = threading.Thread(target=call_groq_api, args=(rms, kurtosis_val))
        bg_thread.start()
        
    # RECOVERY STATE: Allow the machine to heal if the signal goes back down
    elif anomaly_streak == 0 and current_state == "CRITICAL":
        current_state = "NORMAL"
        print("✅ [Subscriber] RECOVERY: Signal back to normal. State reset.", flush=True)

def run_subscriber():
    client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
    client.on_connect = on_connect
    client.on_message = on_message
    client.connect(BROKER_ADDRESS, PORT, 60)
    client.loop_forever()

threading.Thread(target=run_subscriber, daemon=True).start()
time.sleep(2)

# --- 4. PUBLISHER ---
print(f"📡 Streaming data...")
pub_client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
pub_client.connect(BROKER_ADDRESS, PORT, 60)

window_size, step_size = 1024, 512
for i in range(0, len(timeline) - window_size, step_size):
    window = timeline[i:i+window_size]
    rms = float(np.sqrt(np.mean(window**2)))
    kurt = float(kurtosis(window))
    crest = float(np.max(np.abs(window)) / rms if rms > 0 else 0)
    skew_val = float(skew(window))
    payload = {"sensor_id": "MOTOR-01", "features": [rms, kurt, crest, skew_val]}
    pub_client.publish(TOPIC, json.dumps(payload))
    if i % (step_size * 20) == 0:
        print(f"Streaming... RMS: {rms:.3f}")
    time.sleep(0.05)

print("\n✅ Publisher finished.")
print("⏳ Waiting for Groq API Thread to finish...")

if bg_thread is not None:
    bg_thread.join(timeout=15)

print("🏁 Execution Finished.")

In [30]:
import requests
import base64
import json
import getpass
import os

# --- PRE-CONFIGURED FOR YOUR REPO ---
OWNER = "maashg3l4"
REPO = "legacy-asset-maintenance-copilot"
FILE_PATH = "03_mqtt_groq_copilot_pipeline.ipynb"
COMMIT_MESSAGE = "feat: End-to-end MQTT + Anomaly Detection + Groq API Copilot pipeline"

# --- 1. SAVE CURRENT COLAB TO FILE ---
print(f"💾 Saving current notebook to {FILE_PATH}...")
%notebook -e {FILE_PATH}

# --- 2. GET TOKEN ---
GITHUB_TOKEN = getpass.getpass("🔑 Paste your GitHub Token: ")

# --- 3. PUSH TO GITHUB ---
def push_to_github(local_path, repo_path, message, token):
    if not os.path.exists(local_path):
        print(f"❌ Error: File '{local_path}' was not created.")
        return

    with open(local_path, "rb") as f:
        content_base64 = base64.b64encode(f.read()).decode("utf-8")

    api_url = f"https://api.github.com/repos/{OWNER}/{REPO}/contents/{repo_path}"
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28"
    }

    response = requests.get(api_url, headers=headers)
    sha = response.json().get("sha") if response.status_code == 200 else None

    payload = {
        "message": message,
        "content": content_base64
    }
    if sha:
        payload["sha"] = sha

    put_response = requests.put(api_url, headers=headers, data=json.dumps(payload))

    if put_response.status_code in [200, 201]:
        print(f"\n🚀 SUCCESS! File pushed to GitHub.")
        print(f"🔗 View it here: https://github.com/{OWNER}/{REPO}/blob/main/{repo_path}")
    else:
        print(f"\n❌ FAILED. Status: {put_response.status_code}")
        print(put_response.json())

push_to_github(FILE_PATH, FILE_PATH, COMMIT_MESSAGE, GITHUB_TOKEN)

In [31]:
import requests, base64, getpass, os

# YOUR EXACT DETAILS
OWNER = "maashg3l4"
REPO = "legacy-asset-maintenance-copilot"
FILE_PATH = "03_mqtt_groq_copilot_pipeline.ipynb"

# 1. Save notebook
%notebook -e {FILE_PATH}

# 2. Read the file and remove your API key so GitHub allows the upload
with open(FILE_PATH, "r", encoding="utf-8") as f:
    text = f.read()
text = text.replace("YOUR_GROQ_KEY_HERE", "YOUR_GROQ_KEY")
with open(FILE_PATH, "w", encoding="utf-8") as f:
    f.write(text)

# 3. Get Token
token = getpass.getpass("Paste GitHub Token: ")

# 4. Push
url = f"https://api.github.com/repos/{OWNER}/{REPO}/contents/{FILE_PATH}"
headers = {"Authorization": f"Bearer {token}", "Accept": "application/vnd.github+json"}

with open(FILE_PATH, "rb") as f:
    content = base64.b64encode(f.read()).decode("utf-8")

# Check if we need to update an existing file
res = requests.get(url, headers=headers)
sha = res.json().get("sha") if res.status_code == 200 else None

payload = {"message": "feat: complete MQTT pipeline", "content": content}
if sha: payload["sha"] = sha

# Send
put = requests.put(url, headers=headers, json=payload)

if put.status_code in [200, 201]:
    print(f"\n🚀 SUCCESS! View it here: https://github.com/{OWNER}/{REPO}/blob/main/{FILE_PATH}")
else:
    print("FAILED:", put.json())